# Thesis PDF context extraction

Trims **references / bibliography / appendix** heuristically using `thesisdatarepo.pdf_context` (pypdf).

**Environment:** from the repo root, use uv so the package is on the path:

```bash
uv run jupyter lab maks/Analysis_main.ipynb
```

Or: `uv sync` then select the project interpreter (`.venv`).

**Data:** put PDFs in a local folder (e.g. sync from GCS first):

```bash
mkdir -p data/master_thesis_sample
gsutil -m cp "gs://thesis_archive_bucket/dtu_findit/master_thesis/*.pdf" data/master_thesis_sample/  # large; prefer a subset
```


In [ ]:
from pathlib import Path

from thesisdatarepo.pdf_context import FallbackPolicy, process_folder, process_one_pdf

# Prefer repo root whether the kernel cwd is `maks/` or the project root.
_here = Path.cwd().resolve()
REPO_ROOT = _here if (_here / "pyproject.toml").is_file() else (_here / "..").resolve()
INPUT_DIR = REPO_ROOT / "data" / "master_thesis_sample"
OUTPUT_DIR = REPO_ROOT / "data" / "master_thesis_context"
MANIFEST_CSV = REPO_ROOT / "data" / "manifest_context.csv"
MANIFEST_JSON = REPO_ROOT / "data" / "manifest_context.json"

INPUT_DIR, OUTPUT_DIR

In [ ]:
# Batch: all PDFs in INPUT_DIR -> OUTPUT_DIR + manifest (CSV)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

results = process_folder(
    INPUT_DIR,
    OUTPUT_DIR,
    MANIFEST_CSV,
    manifest_format="csv",
    min_page_fraction=0.45,
    policy=FallbackPolicy.KEEP_ALL,
)
len(results), MANIFEST_CSV

In [ ]:
# Optional: JSON manifest
from thesisdatarepo.pdf_context import process_folder

_ = process_folder(
    INPUT_DIR,
    OUTPUT_DIR,
    MANIFEST_JSON,
    manifest_format="json",
    min_page_fraction=0.45,
)


In [ ]:
# Single-file example (first PDF in folder)
pdfs = sorted(INPUT_DIR.glob("*.pdf"))
if pdfs:
    out = OUTPUT_DIR / "_single_sample.pdf"
    r = process_one_pdf(
        pdfs[0],
        out,
        min_page_fraction=0.45,
        policy=FallbackPolicy.KEEP_ALL,
    )
    r
else:
    print("No PDFs in", INPUT_DIR)